# **Data Cleaning**

In [128]:
# Load pandas for data processing and datetime utilities for any date operations.
import pandas as pd

# Set option to display all columns and format float values to two decimal places (to avoid scientific notation).
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [129]:
# Load the raw e-commerce dataset for cleaning.
ecommerce_df = pd.read_csv(r'dataset/raw/ecommerce_dataset_+1m.csv')

## Initial Cleaning

Goals for this section:
- Remove unnecessary / redundant columns
- Inspect data quality (nulls, sample, dtypes)
- Round float columns to 2 decimal places
- Flag logically invalid values

In [130]:
# Display all the columns
ecommerce_df.columns

Index(['order_id', 'order_date', 'order_year', 'order_month', 'order_day',
       'order_hour', 'order_minute', 'order_second', 'is_weekend',
       'order_status', 'return_reason', 'customer_id', 'customer_name',
       'gender', 'age', 'customer_segment', 'country', 'city',
       'customer_loyalty_score', 'total_orders_by_customer',
       'account_creation_date', 'product_id', 'product_name', 'category',
       'sub_category', 'brand', 'product_rating_avg', 'product_reviews_count',
       'stock_quantity', 'unit_price_usd', 'quantity', 'discount_percent',
       'discount_amount_usd', 'total_price_usd', 'cost_usd', 'profit_usd',
       'tax_usd', 'currency', 'payment_method', 'payment_status',
       'installment_plan', 'shipping_method', 'shipping_cost_usd',
       'delivery_days', 'shipping_country', 'warehouse_location',
       'delivery_status', 'rating', 'review_sentiment', 'customer_feedback',
       'coupon_used', 'coupon_code', 'campaign_source', 'device_type',
       'traf

### 1. Remove unnecessary columns

In [131]:

# Eliminate columns that are not relevant to the analysis or contain redundant information.

ecommerce_df = ecommerce_df[
    [
        # TIME & CONTEXT    
        'order_date',
        'order_year',
        'order_month',
        
        # CUSTOMER DEMOGRAPHICS / GEOGRAPHY
        'customer_name',
        'gender',
        'age',
        'customer_segment',
        'country',
        
        # PRODUCT
        'category',
        'sub_category',
        'unit_price_usd',
        'quantity',
        
        # FINANCIALS
        'discount_percent',
        'total_price_usd',
        'profit_usd',
        'profit_margin_percent',
        
        # PAYMENT & SHIPPING
        'payment_method',
        'shipping_method',
        'shipping_cost_usd',
        'delivery_days',
        'shipping_country',
        
        # CUSTOMER BEHAVIOR
        'rating',
        'customer_loyalty_score',
        'coupon_used',
        'session_duration_minutes',
        'pages_visited',
        'abandoned_cart_before',
        
        # RISK & PERFORMANCE
        'fraud_risk_score',
        'device_type',
        
        # MARKETING
        'campaign_source',
        'traffic_source',
    ]
].copy()

### 2. Rename Columns

In [132]:
ecommerce_df = ecommerce_df.rename(
    columns={
        'total_price_usd' : 'revenue_usd',
        'session_duration_minutes' : 'session_duration_min'
    }
)

### 3. Data quality check

In [133]:
ecommerce_df.info()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000123 entries, 0 to 1000122
Data columns (total 31 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   order_date              1000123 non-null  object 
 1   order_year              1000123 non-null  int64  
 2   order_month             1000123 non-null  int64  
 3   customer_name           1000123 non-null  object 
 4   gender                  1000123 non-null  object 
 5   age                     1000123 non-null  int64  
 6   customer_segment        1000123 non-null  object 
 7   country                 1000123 non-null  object 
 8   category                1000123 non-null  object 
 9   sub_category            1000123 non-null  object 
 10  unit_price_usd          1000123 non-null  float64
 11  quantity                1000123 non-null  int64  
 12  discount_percent        1000123 non-null  int64  
 13  revenue_usd             1000123 non-null  float64
 14  pr

In [134]:
# Missing values (%)
print(((ecommerce_df.isnull().sum() / len(ecommerce_df)) * 100).round(2).to_string())

order_date               0.00
order_year               0.00
order_month              0.00
customer_name            0.00
gender                   0.00
age                      0.00
customer_segment         0.00
country                  0.00
category                 0.00
sub_category             0.00
unit_price_usd           0.00
quantity                 0.00
discount_percent         0.00
revenue_usd              0.00
profit_usd               0.00
profit_margin_percent    0.00
payment_method           0.00
shipping_method          0.00
shipping_cost_usd        0.00
delivery_days            0.00
shipping_country         0.00
rating                   0.00
customer_loyalty_score   0.00
coupon_used              0.00
session_duration_min     0.00
pages_visited            0.00
abandoned_cart_before    0.00
fraud_risk_score         0.00
device_type              0.00
campaign_source          0.00
traffic_source           0.00


In [135]:
# Random sample
ecommerce_df.sample(10)

,order_date,order_year,order_month,customer_name,gender,age,customer_segment,country,category,sub_category,unit_price_usd,quantity,discount_percent,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,delivery_days,shipping_country,rating,customer_loyalty_score,coupon_used,session_duration_min,pages_visited,abandoned_cart_before,fraud_risk_score,device_type,campaign_source,traffic_source
12558,2025-03-15 13:46:29.844994,2025,3,Jacob Diaz,Male,39,Regular,Spain,Home,Bedding,149.02,1,10,134.12,30.06,22.41,PayPal,Express,21.27,7,Spain,5,93.30,Yes,36.00,6,Yes,6.90,Desktop,Facebook,Email
906807,2025-03-30 03:23:56.678749,2025,3,Robert Lewis,Male,71,Premium,Italy,Health,Fitness,34.32,4,5,130.42,39.86,30.56,Bank Transfer,Economy,9.53,5,Italy,3,35.60,No,20.10,2,Yes,73.60,Tablet,Facebook,Direct
114927,2024-04-22 15:42:26.667073,2024,4,Kimberly Stokes,Female,66,Regular,Netherlands,Home,Kitchen,289.78,5,15,"1,231.57",304.57,24.73,Apple Pay,Standard,20.78,13,Netherlands,2,78.00,No,24.30,4,Yes,79.60,Desktop,Google Ads,Email
394702,2025-11-09 10:59:50.453833,2025,11,Jay Gonzalez,Male,55,Premium,Spain,Clothing,Womens Wear,44.45,3,10,120.01,45.70,38.08,Credit Card,Standard,0.56,2,Spain,2,35.30,Yes,55.20,16,Yes,16.40,Tablet,Google Ads,Search
553645,2024-08-28 16:39:21.745125,2024,8,Angela Dunlap,Female,48,Regular,Belgium,Home,Decor,185.53,4,20,593.70,288.30,48.56,Apple Pay,Express,16.14,8,Belgium,2,74.70,Yes,27.80,18,Yes,73.20,Mobile,Instagram,Social
445417,2025-01-21 17:19:44.013960,2025,1,Connor Daniels,Male,27,Regular,United Kingdom,Health,Fitness,184.40,1,20,147.52,65.97,44.72,Credit Card,Express,0.73,11,United Kingdom,5,19.50,No,43.80,3,No,0.00,Mobile,Email,Direct
893765,2025-12-18 23:55:18.926712,2025,12,Karen Garcia,Female,47,Premium,Australia,Sports,Accessories,93.04,4,0,372.16,115.60,31.06,Bank Transfer,Standard,17.01,13,Australia,4,27.30,No,53.00,4,No,92.30,Desktop,Google Ads,Social
64376,2024-11-19 23:07:17.136283,2024,11,Ryan Bryant,Male,52,Regular,France,Sports,Sports Wear,55.18,2,5,104.84,30.16,28.77,Bank Transfer,Express,20.61,14,France,4,22.80,Yes,35.50,12,Yes,43.40,Desktop,Email,Search
812302,2024-10-31 01:52:20.469611,2024,10,Shannon Gonzalez,Female,53,Premium,Italy,Electronics,Laptops,296.28,1,10,266.65,119.87,44.95,Debit Card,Next Day,8.90,14,Italy,1,4.50,No,54.10,5,Yes,61.80,Desktop,Google Ads,Search
190121,2025-04-24 20:45:48.129354,2025,4,Kimberly Johnson,Female,28,Regular,Germany,Sports,Gym Equipment,164.49,5,10,740.20,352.50,47.62,Credit Card,Express,17.64,3,Germany,1,74.30,No,46.50,8,Yes,75.50,Tablet,Email,Social


### 4. Round float columns to 2 decimal places

In [136]:
# Round float values to two decimal places for cleaner output.
float_cols = ecommerce_df.select_dtypes('float')

for col in float_cols.columns:
    ecommerce_df[col] = ecommerce_df[col].round(2)

### 5. Flag logically invalid values

In [137]:
# 1. Age
invalid_age = ecommerce_df[(ecommerce_df['age'] < 0) | (ecommerce_df['age'] > 120)]

# 2. Quantity
invalid_quantity = ecommerce_df[ecommerce_df['quantity'] <= 0]

# 3. Discount
invalid_discount = ecommerce_df[
    (ecommerce_df['discount_percent'] < 0) | (ecommerce_df['discount_percent'] > 100)
]

# 4. Rating (fix this!)
invalid_rating = ecommerce_df[(ecommerce_df['rating'] < 1) | (ecommerce_df['rating'] > 5)]

# 5. Delivery days
invalid_delivery = ecommerce_df[
    (ecommerce_df['delivery_days'] < 0) | (ecommerce_df['delivery_days'] > 60)
]

# 6. Monetary values
invalid_money = ecommerce_df[
    (ecommerce_df['unit_price_usd'] < 0) |
    (ecommerce_df['revenue_usd'] < 0) |
    (ecommerce_df['shipping_cost_usd'] < 0)
]

# 7. Fraud score
invalid_fraud = ecommerce_df[
    (ecommerce_df['fraud_risk_score'] < 0) | (ecommerce_df['fraud_risk_score'] > 100)
]

# Print summary
print(f'Invalid Age        : {len(invalid_age)}')
print(f'Invalid Quantity   : {len(invalid_quantity)}')
print(f'Invalid Discount   : {len(invalid_discount)}')
print(f'Invalid Rating     : {len(invalid_rating)}')
print(f'Invalid Delivery   : {len(invalid_delivery)}')
print(f'Invalid Money      : {len(invalid_money)}')
print(f'Invalid Fraud      : {len(invalid_fraud)}')

Invalid Age        : 0
Invalid Quantity   : 0
Invalid Discount   : 0
Invalid Rating     : 0
Invalid Delivery   : 0
Invalid Money      : 0
Invalid Fraud      : 0


### 6. Save cleaned dataset

In [138]:
# Save the cleaned dataset to a new CSV file.
ecommerce_df.to_csv("dataset/cleaned/ecommerce_cleaned.csv", index=False)